# TAD y listas enlazadas · Unidad 2, Tema 1
**Estudiante:** Joseph Damian Pardo Gualan · Ciencia de Datos e Inteligencia Artificial · UNACH

Cada ejercicio está en una celda independiente. Al ejecutarla se abre un menú interactivo y todos los datos se ingresan manualmente por teclado (sin `assert`).

## Ejercicio 1: Historial de despachos finalizados (lista enlazada simple)

In [ ]:
"""Ejercicio 1: Historial de despachos finalizados (lista enlazada simple).
Todos los datos se ingresan manualmente por teclado."""
from datetime import datetime
from decimal import Decimal, InvalidOperation


# ---------- Entrada de datos validada (sin asserts) ----------
def leer_entero(mensaje, minimo=1):
    while True:
        texto = input(mensaje).strip()
        try:
            valor = int(texto)
            if valor >= minimo:
                return valor
            print(f"  [!] Debe ser un entero mayor o igual a {minimo}.")
        except ValueError:
            print("  [!] Ingrese un número entero válido.")


def leer_texto(mensaje):
    while True:
        texto = input(mensaje).strip()
        if texto:
            return texto
        print("  [!] El campo no puede quedar vacío.")


def leer_fecha(mensaje):
    while True:
        texto = input(mensaje).strip()
        try:
            datetime.strptime(texto, "%d/%m/%Y")
            return texto
        except ValueError:
            print("  [!] Use el formato DD/MM/AAAA con una fecha real.")


def leer_decimal(mensaje, permitir_cero=False):
    while True:
        texto = input(mensaje).strip().replace(",", ".")
        try:
            valor = Decimal(texto)
            if valor.is_finite() and (valor > 0 or (permitir_cero and valor == 0)):
                return valor.quantize(Decimal("0.01"))
            print("  [!] El monto debe ser positivo.")
        except InvalidOperation:
            print("  [!] Ingrese un valor numérico (ej. 350.50).")


# ---------- Modelo de datos y nodo ----------
class RegistroDespacho:
    def __init__(self, id_registro, nombre_transportista, fecha, monto_flete):
        self.id_registro = id_registro
        self.nombre_transportista = nombre_transportista
        self.fecha = fecha
        self.monto_flete = monto_flete

    def __str__(self):
        return (f"ID {self.id_registro:<4}| {self.nombre_transportista:<18}| "
                f"{self.fecha} | $ {self.monto_flete:>9.2f}")


class NodoSimple:
    def __init__(self, dato):
        self.dato = dato      # objeto RegistroDespacho
        self.sig = None       # puntero al siguiente nodo


# ---------- TAD: lista enlazada simple ----------
class HistorialDespachos:
    def __init__(self):
        self.inicio = None
        self.fin = None       # permite insertar al final en O(1)
        self.cantidad = 0

    def existe_id(self, id_registro):
        actual = self.inicio
        while actual is not None:
            if actual.dato.id_registro == id_registro:
                return True
            actual = actual.sig
        return False

    def insertar_final(self, registro):
        nuevo = NodoSimple(registro)
        if self.inicio is None:          # lista vacía
            self.inicio = nuevo
        else:                            # se enlaza detrás del último
            self.fin.sig = nuevo
        self.fin = nuevo
        self.cantidad += 1

    def mostrar_historial(self):
        print("\n=== HISTORIAL DE DESPACHOS FINALIZADOS ===")
        if self.inicio is None:
            print("Sin despachos registrados.")
            return
        actual = self.inicio
        while actual is not None:        # recorrido simple hasta None
            print(actual.dato)
            actual = actual.sig
        print(f"Total de despachos: {self.cantidad}")

    def filtrar_por_monto(self, valor_minimo):
        print(f"\n=== DESPACHOS CON FLETE MAYOR A $ {valor_minimo:.2f} ===")
        actual = self.inicio
        encontrados = 0
        while actual is not None:
            if actual.dato.monto_flete > valor_minimo:
                print(actual.dato)
                encontrados += 1
            actual = actual.sig
        if encontrados == 0:
            print("Ningún despacho supera ese monto.")
        else:
            print(f"Coincidencias: {encontrados} de {self.cantidad}")


# ---------- Programa principal (menú interactivo) ----------
def registrar_despacho(historial):
    print("\n--- Nuevo despacho ---")
    while True:
        id_registro = leer_entero("ID del registro: ")
        if not historial.existe_id(id_registro):
            break
        print("  [!] Ese ID ya existe en el historial.")
    nombre = leer_texto("Nombre del transportista: ")
    fecha = leer_fecha("Fecha (DD/MM/AAAA): ")
    monto = leer_decimal("Monto del flete ($): ")
    historial.insertar_final(RegistroDespacho(id_registro, nombre, fecha, monto))
    print(f"  [OK] Despacho {id_registro} insertado al final de la lista.")


def menu():
    historial = HistorialDespachos()
    while True:
        print("\n===== EJERCICIO 1 · LISTA SIMPLE =====")
        print("[1] Registrar despacho  [2] Mostrar historial")
        print("[3] Filtrar por monto   [4] Salir")
        opcion = input("Seleccione una opción: ").strip()
        if opcion == "1":
            registrar_despacho(historial)
        elif opcion == "2":
            historial.mostrar_historial()
        elif opcion == "3":
            minimo = leer_decimal("Monto mínimo para la auditoría ($): ",
                                  permitir_cero=True)
            historial.filtrar_por_monto(minimo)
        elif opcion == "4":
            print("Fin del módulo de despachos.")
            break
        else:
            print("  [!] Opción no válida.")


if __name__ == "__main__":
    menu()


## Ejercicio 2: Zona de triaje y espera de carga (lista doblemente enlazada)

In [ ]:
"""Ejercicio 2: Zona de triaje y espera de carga (lista doblemente enlazada).
Todos los datos se ingresan manualmente por teclado."""
from datetime import datetime


# ---------- Entrada de datos validada (sin asserts) ----------
def leer_entero(mensaje, minimo=1):
    while True:
        texto = input(mensaje).strip()
        try:
            valor = int(texto)
            if valor >= minimo:
                return valor
            print(f"  [!] Debe ser un entero mayor o igual a {minimo}.")
        except ValueError:
            print("  [!] Ingrese un número entero válido.")


def leer_texto(mensaje):
    while True:
        texto = input(mensaje).strip()
        if texto:
            return texto
        print("  [!] El campo no puede quedar vacío.")


def leer_hora(mensaje):
    while True:
        texto = input(mensaje).strip()
        try:
            datetime.strptime(texto, "%H:%M")
            return texto
        except ValueError:
            print("  [!] Use el formato HH:MM de 24 horas (ej. 07:45).")


def leer_tipo_carga():
    while True:
        print("Tipo de carga: 1) Perecederos  2) Frágil  3) General")
        opcion = input("Seleccione: ").strip()
        if opcion == "1":
            return "Perecederos"
        if opcion == "2":
            return "Frágil"
        if opcion == "3":
            return "General"
        print("  [!] Elija 1, 2 o 3.")


# ---------- Modelo de datos y nodo ----------
class VehiculoEspera:
    def __init__(self, numero_ticket, placa_vehiculo, hora_llegada, tipo_carga):
        self.numero_ticket = numero_ticket
        self.placa_vehiculo = placa_vehiculo.upper()
        self.hora_llegada = hora_llegada
        self.tipo_carga = tipo_carga

    def __str__(self):
        return (f"Ticket {self.numero_ticket} | {self.placa_vehiculo:<9}| "
                f"{self.hora_llegada} | {self.tipo_carga}")


class NodoDoble:
    def __init__(self, dato):
        self.dato = dato
        self.ant = None       # puntero al nodo anterior
        self.sig = None       # puntero al nodo siguiente


# ---------- TAD: lista doblemente enlazada ----------
class FilaEspera:
    def __init__(self):
        self.inicio = None
        self.fin = None
        self.cantidad = 0

    def buscar_nodo(self, numero_ticket):
        actual = self.inicio
        while actual is not None:
            if actual.dato.numero_ticket == numero_ticket:
                return actual
            actual = actual.sig
        return None

    def registrar_llegada(self, vehiculo):
        nuevo = NodoDoble(vehiculo)
        if self.inicio is None:          # primer camión de la bahía
            self.inicio = nuevo
        else:                            # enlace doble con el antiguo fin
            nuevo.ant = self.fin
            self.fin.sig = nuevo
        self.fin = nuevo
        self.cantidad += 1

    def mostrar_fila(self, adelante=True):
        titulo = "INICIO -> FIN" if adelante else "FIN -> INICIO"
        print(f"\n=== FILA DE ESPERA ({titulo}) ===")
        if self.inicio is None:
            print("Sin vehículos en espera.")
            return
        actual = self.inicio if adelante else self.fin
        posicion = 1
        while actual is not None:
            print(f"{posicion}. {actual.dato}")
            actual = actual.sig if adelante else actual.ant
            posicion += 1

    def navegar_fila(self, numero_ticket):
        actual = self.buscar_nodo(numero_ticket)
        if actual is None:
            print(f"  [!] No existe el ticket {numero_ticket}.")
            return
        while True:
            anterior = actual.ant.dato if actual.ant else "Ninguno (inicio de fila)"
            siguiente = actual.sig.dato if actual.sig else "Ninguno (fin de fila)"
            print(f"\n  ANTERIOR    : {anterior}")
            print(f"> SELECCIONADO: {actual.dato}")
            print(f"  SIGUIENTE   : {siguiente}")
            paso = input("[S] siguiente  [A] anterior  [X] salir: ").strip().upper()
            if paso == "S" and actual.sig is not None:
                actual = actual.sig
            elif paso == "A" and actual.ant is not None:
                actual = actual.ant
            elif paso == "X":
                break
            else:
                print("  [!] Movimiento no disponible en ese sentido.")

    def mover_a_inicio(self, numero_ticket):
        nodo = self.buscar_nodo(numero_ticket)
        if nodo is None:
            print(f"  [!] No existe el ticket {numero_ticket}.")
            return
        if nodo is self.inicio:
            print("  [i] Ese vehículo ya está en la cabeza de la fila.")
            return
        # 1) Desenlazar: el anterior y el siguiente se conectan entre sí
        nodo.ant.sig = nodo.sig
        if nodo.sig is not None:
            nodo.sig.ant = nodo.ant
        else:                            # era el último: fin retrocede
            self.fin = nodo.ant
        # 2) Reenlazar el MISMO nodo delante del inicio actual
        nodo.ant = None
        nodo.sig = self.inicio
        self.inicio.ant = nodo
        self.inicio = nodo
        print(f"  [OK] Pase directo: ticket {numero_ticket} movido a la cabeza.")

    def verificar_enlaces(self):
        """Comprueba con if (no assert) que ant y sig sean coherentes."""
        errores = 0
        contador = 0
        actual = self.inicio
        if actual is not None and actual.ant is not None:
            errores += 1
        while actual is not None:
            contador += 1
            if actual.sig is not None and actual.sig.ant is not actual:
                errores += 1
            if actual.sig is None and actual is not self.fin:
                errores += 1
            actual = actual.sig
        if contador != self.cantidad:
            errores += 1
        if errores == 0:
            print(f"  [OK] {contador} nodo(s): punteros ant/sig coherentes.")
        else:
            print(f"  [X] Se detectaron {errores} inconsistencias.")


# ---------- Programa principal (menú interactivo) ----------
def registrar(fila):
    print("\n--- Llegada de vehículo ---")
    while True:
        ticket = leer_entero("Número de ticket (ej. 101): ")
        if fila.buscar_nodo(ticket) is None:
            break
        print("  [!] Ese ticket ya está en la fila.")
    placa = leer_texto("Placa del vehículo: ")
    hora = leer_hora("Hora de llegada (HH:MM): ")
    carga = leer_tipo_carga()
    fila.registrar_llegada(VehiculoEspera(ticket, placa, hora, carga))
    print(f"  [OK] Ticket {ticket} registrado al final de la fila.")


def menu():
    fila = FilaEspera()
    while True:
        print("\n===== EJERCICIO 2 · LISTA DOBLE =====")
        print("[1] Registrar llegada   [2] Fila inicio->fin  [3] Fila fin->inicio")
        print("[4] Navegar fila        [5] Pase directo      [6] Verificar punteros")
        print("[7] Salir")
        opcion = input("Seleccione una opción: ").strip()
        if opcion == "1":
            registrar(fila)
        elif opcion == "2":
            fila.mostrar_fila(adelante=True)
        elif opcion == "3":
            fila.mostrar_fila(adelante=False)
        elif opcion == "4":
            fila.navegar_fila(leer_entero("Ticket a seleccionar: "))
        elif opcion == "5":
            fila.mover_a_inicio(leer_entero("Ticket con carga urgente: "))
        elif opcion == "6":
            fila.verificar_enlaces()
        elif opcion == "7":
            print("Fin del módulo de espera de carga.")
            break
        else:
            print("  [!] Opción no válida.")


if __name__ == "__main__":
    menu()


## Ejercicio 3: Monitoreo cíclico de rutas (lista doble circular)

In [ ]:
"""Ejercicio 3: Monitoreo cíclico de rutas (lista doble circular).
Todos los datos se ingresan manualmente por teclado."""


# ---------- Entrada de datos validada (sin asserts) ----------
def leer_entero(mensaje, minimo=0):
    while True:
        texto = input(mensaje).strip()
        try:
            valor = int(texto)
            if valor >= minimo:
                return valor
            print(f"  [!] Debe ser un entero mayor o igual a {minimo}.")
        except ValueError:
            print("  [!] Ingrese un número entero válido.")


def leer_texto(mensaje):
    while True:
        texto = input(mensaje).strip()
        if texto:
            return texto
        print("  [!] El campo no puede quedar vacío.")


def leer_estado():
    while True:
        print("Estado: 1) En Ruta  2) Detenido  3) Alerta de Desvío")
        opcion = input("Seleccione: ").strip()
        if opcion == "1":
            return "En Ruta"
        if opcion == "2":
            return "Detenido"
        if opcion == "3":
            return "Alerta de Desvío"
        print("  [!] Elija 1, 2 o 3.")


# ---------- Modelo de datos y nodo ----------
class MonitoreoRuta:
    def __init__(self, id_unidad, nombre_chofer, velocidad_actual, estado_ruta):
        self.id_unidad = id_unidad
        self.nombre_chofer = nombre_chofer
        self.velocidad_actual = velocidad_actual
        self.estado_ruta = estado_ruta

    def __str__(self):
        return (f"Unidad {self.id_unidad} | {self.nombre_chofer:<10}| "
                f"{self.velocidad_actual:>3} km/h | {self.estado_ruta}")


class NodoCircular:
    def __init__(self, dato):
        self.dato = dato
        self.sig = None
        self.ant = None


# ---------- TAD: lista doble circular ----------
class MonitorCircular:
    def __init__(self):
        self.inicio = None    # no existe 'fin': el último es inicio.ant
        self.cantidad = 0

    def buscar_nodo(self, id_unidad):
        if self.inicio is None:
            return None
        actual = self.inicio
        while True:                      # se detiene al volver al inicio
            if actual.dato.id_unidad == id_unidad:
                return actual
            actual = actual.sig
            if actual is self.inicio:
                return None

    def conectar_unidad(self, unidad):
        nuevo = NodoCircular(unidad)
        if self.inicio is None:          # una sola unidad: se apunta a sí misma
            nuevo.sig = nuevo
            nuevo.ant = nuevo
            self.inicio = nuevo
        else:                            # se inserta entre el último y el inicio
            ultimo = self.inicio.ant
            nuevo.ant = ultimo
            nuevo.sig = self.inicio
            ultimo.sig = nuevo
            self.inicio.ant = nuevo
        self.cantidad += 1

    def desconectar_unidad(self, id_unidad):
        nodo = self.buscar_nodo(id_unidad)
        if nodo is None:
            print(f"  [!] La unidad {id_unidad} no está en monitoreo.")
            return
        if nodo.sig is nodo:             # era la única unidad
            self.inicio = None
        else:                            # se cierra el ciclo sin el nodo
            nodo.ant.sig = nodo.sig
            nodo.sig.ant = nodo.ant
            if nodo is self.inicio:
                self.inicio = nodo.sig
        nodo.sig = None                  # se aísla el nodo retirado
        nodo.ant = None
        self.cantidad -= 1
        print(f"  [OK] Unidad {id_unidad} llegó a destino y salió del ciclo.")

    def carrusel(self, vueltas, adelante=True):
        if self.inicio is None:
            print("No hay unidades en carretera.")
            return
        sentido = "horario (sig)" if adelante else "antihorario (ant)"
        print(f"\n=== PANEL CÍCLICO · {vueltas} vuelta(s) · sentido {sentido} ===")
        actual = self.inicio if adelante else self.inicio.ant
        for paso in range(1, self.cantidad * vueltas + 1):
            print(f"[{paso:02d}] {actual.dato}")
            actual = actual.sig if adelante else actual.ant
            if actual is self.inicio and adelante and paso < self.cantidad * vueltas:
                print("     ---- regreso automático a la primera unidad ----")

    def actualizar_telemetria(self, id_unidad, velocidad, estado):
        nodo = self.buscar_nodo(id_unidad)
        if nodo is None:
            print(f"  [!] La unidad {id_unidad} no está en monitoreo.")
            return
        nodo.dato.velocidad_actual = velocidad
        nodo.dato.estado_ruta = estado
        print(f"  [OK] Telemetría de la unidad {id_unidad} actualizada.")

    # ----- PARTE PERSONALIZADA: semáforo de riesgo en el carrusel -----
    def panel_alertas(self, limite):
        if self.inicio is None:
            print("No hay unidades en carretera.")
            return
        print(f"\n=== SEMÁFORO DE RIESGO (límite {limite} km/h) ===")
        criticas = altas = medias = 0
        actual = self.inicio
        for _ in range(self.cantidad):   # exactamente una vuelta
            dato = actual.dato
            exceso = dato.velocidad_actual > limite
            desvio = dato.estado_ruta == "Alerta de Desvío"
            if exceso and desvio:
                nivel, accion = "ROJO", "llamar al chofer y bloquear ruta"
                criticas += 1
            elif exceso:
                extra = dato.velocidad_actual - limite
                nivel, accion = "NARANJA", f"reducir {extra} km/h de inmediato"
                altas += 1
            elif desvio:
                nivel, accion = "NARANJA", "verificar ruta autorizada por GPS"
                altas += 1
            elif dato.estado_ruta == "Detenido":
                nivel, accion = "AMARILLO", "confirmar motivo de la parada"
                medias += 1
            else:
                nivel, accion = "VERDE", "supervisión normal"
            print(f"{nivel:<8} | Unidad {dato.id_unidad} | {accion}")
            actual = actual.sig
        print(f"Resumen -> rojo: {criticas} | naranja: {altas} | amarillo: {medias}")

    def verificar_circularidad(self):
        """Comprueba con if (no assert) que no exista ningún None."""
        if self.inicio is None:
            print("  [i] Ciclo vacío: inicio = None, cantidad = 0.")
            return
        errores = 0
        actual = self.inicio
        for _ in range(self.cantidad):
            if actual.sig is None or actual.ant is None or actual.sig.ant is not actual:
                errores += 1
            actual = actual.sig
        if actual is not self.inicio:    # tras n pasos debe volver al inicio
            errores += 1
        if errores == 0:
            print(f"  [OK] {self.cantidad} nodo(s): ciclo cerrado en ambos sentidos.")
        else:
            print(f"  [X] Se detectaron {errores} inconsistencias.")


# ---------- Programa principal (menú interactivo) ----------
def conectar(monitor):
    print("\n--- Unidad que sale de la central ---")
    while True:
        id_unidad = leer_entero("ID de la unidad: ", minimo=1)
        if monitor.buscar_nodo(id_unidad) is None:
            break
        print("  [!] Esa unidad ya está en monitoreo.")
    chofer = leer_texto("Nombre del chofer: ")
    velocidad = leer_entero("Velocidad actual (km/h): ")
    estado = leer_estado()
    monitor.conectar_unidad(MonitoreoRuta(id_unidad, chofer, velocidad, estado))
    print(f"  [OK] Unidad {id_unidad} conectada al ciclo de monitoreo.")


def menu():
    monitor = MonitorCircular()
    while True:
        print("\n===== EJERCICIO 3 · LISTA DOBLE CIRCULAR =====")
        print("[1] Conectar unidad      [2] Desconectar unidad  [3] Ver carrusel")
        print("[4] Actualizar telemetría [5] Semáforo de riesgo [6] Verificar ciclo")
        print("[7] Salir")
        opcion = input("Seleccione una opción: ").strip()
        if opcion == "1":
            conectar(monitor)
        elif opcion == "2":
            monitor.desconectar_unidad(leer_entero("ID a desconectar: ", 1))
        elif opcion == "3":
            vueltas = leer_entero("Número de vueltas: ", minimo=1)
            sentido = input("Sentido [A]delante / [R]etroceso: ").strip().upper()
            monitor.carrusel(vueltas, adelante=(sentido != "R"))
        elif opcion == "4":
            id_unidad = leer_entero("ID de la unidad: ", 1)
            velocidad = leer_entero("Nueva velocidad (km/h): ")
            monitor.actualizar_telemetria(id_unidad, velocidad, leer_estado())
        elif opcion == "5":
            monitor.panel_alertas(leer_entero("Límite de velocidad (km/h): ", 1))
        elif opcion == "6":
            monitor.verificar_circularidad()
        elif opcion == "7":
            print("Fin del módulo de monitoreo.")
            break
        else:
            print("  [!] Opción no válida.")


if __name__ == "__main__":
    menu()
